# Radiographic BPD Likelihood Score - inference notebook

End-to-end inference for the **radiographic bronchopulmonary dysplasia (BPD) likelihood score**:
lung field segmentation with a U-Net, extraction of the left and right lung fields, cropping of
three patches per lung field, and scoring with an EfficientNet-B0 based classifier.

**Research use only. This is not a medical device and must not be used for diagnosis or
treatment decisions.**

Environment used in the study: Python 3.8.10, TensorFlow 2.13.0 (Keras 2.13.1).
See `README.md` for installation, input/output specifications and caveats.

Run the cells in order.


## 1. Load the chest radiographs

Images are read as grayscale, resized to 256 x 256 and scaled to `[0, 1]`.


In [ ]:
import os
import cv2
import numpy as np

# ---------------------------------------------------------------------------
# Paths - edit these before running
# ---------------------------------------------------------------------------
input_dir = "PATH_to_input"                 # chest radiographs to be scored
segmentation_dir = "PATH_to_segmentation"   # output: segmentation QC figures
regions_dir = "PATH_to_regions"             # output: extracted lung field images
output_csv = "PATH_to_output/output.csv"    # output: lung field-level scores

segmentation_model = "PATH_to_MODEL/finetuning_keras_20260624"
scoring_model = "PATH_to_MODEL/patch_cropping_20260714.keras"

# Create the output directories if they do not exist yet.
for _d in (segmentation_dir, regions_dir, os.path.dirname(output_csv)):
    if _d:
        os.makedirs(_d, exist_ok=True)


def get_image_data(image_path):
    valid_extensions = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")

    images = sorted(
        os.path.join(image_path, fname)
        for fname in os.listdir(image_path)
        if fname.lower().endswith(valid_extensions)
    )

    return images


def load_data(image_files, target_size=(256, 256)):
    images = []
    for file in image_files:
        img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"The image could not be loaded.: {file}")
        img = cv2.resize(img, target_size)
        images.append(img)

    images = np.array(images, dtype=np.float32) / 255.0
    images = np.expand_dims(images, axis=-1)

    return images

image_files = get_image_data(input_dir)
print(f'Found {len(image_files)} images')

if len(image_files) == 0:
    raise ValueError("No images found. Please check the paths and ensure that the directories contain image files.")

images = load_data(image_files)

## 2. Lung field segmentation (U-Net)

The custom loss and metric functions must be supplied as `custom_objects` because they were used when the model was trained.


In [ ]:
import tensorflow as tf

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

model = tf.keras.models.load_model(
    segmentation_model,
    custom_objects={'dice_coef': dice_coef, 'dice_loss': dice_loss, 'bce_dice_loss': bce_dice_loss}
)

# Pediction
preds = model.predict(images, verbose=1)
preds_t = (preds > 0.5).astype(np.uint8)

## 3. Segmentation quality control

Saves an original / mask / overlay figure per radiograph to `segmentation_dir`. Inspect these figures before interpreting the scores.


In [ ]:
import os
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

def plot_sample(X, binary_preds, image_files, save_dir, wait_time=1):
    os.makedirs(save_dir, exist_ok=True)

    for i in range(len(X)):
        fig, axs = plt.subplots(1, 3, figsize=(9, 4))
        filename = os.path.splitext(os.path.basename(image_files[i]))[0]

        # Original
        axs[0].imshow(X[i].squeeze(), cmap='gray')
        axs[0].set_title(filename)
        axs[0].axis('off')

        # Predicted mask. NOTE: the fine-tuned U-Net outputs high values for the
        # NON-lung region, so the white area here is everything except the lungs.
        axs[1].imshow(binary_preds[i].squeeze(), cmap='gray')
        axs[1].set_title('Predicted mask')
        axs[1].axis('off')

        # overlay
        axs[2].imshow(X[i].squeeze(), cmap='gray')
        axs[2].imshow(binary_preds[i].squeeze(), cmap='Reds', alpha=0.5)
        axs[2].set_title('Overlay')
        axs[2].axis('off')

        save_path = os.path.join(save_dir, f"segmentation_{filename}.png")
        fig.savefig(save_path)

        plt.tight_layout()
        clear_output(wait=True)
        display(fig)
        plt.close(fig)

        # time.sleep(wait_time)

# Display and save the segmentation QC figures.
# Inspect these before interpreting any score: marked rotation, inappropriate
# exposure or extensive overlying devices may impair lung field segmentation.
plot_sample(images, preds_t, image_files, segmentation_dir)

## 4. Extract the left and right lung fields

The two largest connected components of the (inverted) mask are kept and assigned to `right` / `left` by their x-centroid; on a chest radiograph the anatomical right lung appears on the left of the image.


In [ ]:
import os
import numpy as np
from scipy import ndimage as ndi
from PIL import Image


def extract_regions(img, mask, image_idx, image_files, min_size):

    mask = np.squeeze(mask)
    labeled, n_labels = ndi.label(mask)
    objects = ndi.find_objects(labeled)

    candidates = []
    for i, sl in enumerate(objects):
        if sl is None: continue

        region_mask = (labeled[sl] == i + 1)
        if region_mask.sum() < min_size: continue

        # Center of gravity
        ys, xs = np.where(region_mask)
        y_center = ys.mean() + sl[0].start
        x_center = xs.mean() + sl[1].start

        region_img = img[sl].copy()
        region_img[~region_mask] = 0

        candidates.append({
            'image': region_img,
            'mask': region_mask,
            'bbox': sl,
            'size': int(region_mask.sum()),
            'x_center': x_center,
            'y_center': y_center,
            'source_image_idx': image_idx,
            'filename': os.path.splitext(os.path.basename(image_files[image_idx]))[0],
        })

    if len(candidates) == 0:
        return []

    if len(candidates) == 1:
        candidates[0]['side'] = 'unknown'
        return candidates

    # Top 2 locations by area
    candidates.sort(key=lambda x: x['size'], reverse=True)
    candidates = candidates[:2]

    # Left and right recognition
    candidates.sort(key=lambda x: x['x_center'])
    candidates[0]['side'] = 'right' # Since it's an X-ray, the left and right sides are reversed.
    candidates[1]['side'] = 'left'
    return candidates


def save_regions(regions_all, save_dir, normalize=True):
    os.makedirs(save_dir, exist_ok=True)
    
    for idx, r in enumerate(regions_all):
        img_array = r['image']

        if img_array.dtype != np.uint8:
            if normalize:
                # normalization
                img_save = (255 * (img_array - img_array.min())
                            / (img_array.max() - img_array.min() + 1e-8)).astype(np.uint8)
            else:
                img_save = np.clip(img_array * 255, 0, 255).astype(np.uint8)

        else:
            img_save = img_array

        img_save = img_save.squeeze()        
        if normalize:
            filename = f"{r['filename']}_{r['side']}.png"
        else:
            filename = f"{r['filename']}_{r['side']}_raw.png"
        filepath = os.path.join(save_dir, filename)
        
        Image.fromarray(img_save).save(filepath)


all_regions = []

for i in range(len(images)):
    img = images[i]
    mask = preds_t[i].astype(bool)

    # The fine-tuned U-Net outputs high values for the NON-lung region, so the
    # prediction is inverted here to obtain the lung fields themselves.
    # Do not remove this inversion: without it the background forms a single
    # connected component and the left and right lung fields cannot be separated.
    mask = ~mask

    regions = extract_regions(img, mask, image_idx=i, image_files=image_files, min_size=300)
    all_regions.extend(regions)

print(f"{len(all_regions)} lung-field images were saved.")

# Based on the segmentation results, extract and save the left and right lungs.
save_regions(all_regions, regions_dir, normalize=True)

## 5. Patch cropping and model input helpers

Three 48 x 48 patches are cropped around one-sixth, one-half and five-sixths of the lung field height, with the position searched to maximise the lung area inside each patch. Patches are resized to 192 x 192, stacked to 3 channels and scaled to `[0, 1]` (the model applies `efficientnet.preprocess_input(x * 255.0)` internally).


In [ ]:
import numpy as np
import tensorflow as tf
import re

IMG_H = 192
IMG_W = 192

def patches_to_input(patches):
    imgs = []
    for patch in patches:
        img = np.stack([patch, patch, patch], axis=-1)
        img = tf.image.resize(img, (IMG_H, IMG_W)).numpy()
        img = img.astype(np.float32) / 255.
        imgs.append(img)

    return {
        "upper": np.expand_dims(imgs[0], axis=0),
        "middle": np.expand_dims(imgs[1], axis=0),
        "lower": np.expand_dims(imgs[2], axis=0),
    }


def predict_lung(model, img):
    patches = crop_three_patches_predict(img, patch_size=48, y_search=12, x_search=6)

    if len(patches) != 3:
        return np.nan

    x = patches_to_input(patches)
    pred = model.predict(x, verbose=0)

    return float(pred[0, 0])


def extract_side(fname):
    m = re.search(r"(left|right)", fname)
    return m.group(1) if m else None


def crop_three_patches_predict(img, patch_size, y_search, x_search,):
    H, W = img.shape

    if H < patch_size or W < patch_size:
        return []

    centers = [int(H * 1 / 6), int(H * 3 / 6), int(H * 5 / 6)]
    patches = []

    for yc in centers:
        best_patch = None
        best_ratio = -1

        y0 = yc - patch_size // 2
        y0 = np.clip(y0, 0, H - patch_size)

        for dy in range(-y_search, y_search + 1):
            y = int(np.clip(y0 + dy, 0, H - patch_size))
            band = img[y:y + patch_size]
            ys, xs = np.where(band > 0)
            if len(xs) == 0: continue
            center_x = int(np.median(xs))

            for dx in range(-x_search, x_search + 1):
                x = center_x - patch_size // 2 + dx
                x = int(np.clip(x, 0, W - patch_size))
                patch = img[y:y + patch_size, x:x + patch_size]
                lung_ratio = np.mean(patch > 0)

                if lung_ratio > best_ratio:
                    best_ratio = lung_ratio
                    best_patch = patch.copy()


        if best_patch is None:
            y = max(0, min(H - patch_size, y0))
            x = max(0, (W - patch_size) // 2)
            best_patch = img[y:y + patch_size, x:x + patch_size].copy()

        patches.append(best_patch)

    return patches

## 6. Lung field-level scoring

Writes one row per lung field to `output_csv`.


In [ ]:
import glob
import os
import cv2
import pandas as pd

# Classification threshold prespecified in the study, derived by the Youden index
# from the lung field-level ROC analysis on the model development test set.
SCORE_THRESHOLD = 0.554

# Load the scoring model. safe_mode=False is required because the model contains
# Lambda layers; only load model files from a source you trust.
model = tf.keras.models.load_model(scoring_model, safe_mode=False, compile=False)


image_files = sorted(glob.glob(os.path.join(regions_dir, "*.png")))
print(f"{len(image_files)} images found.")

scores = []

for file in image_files:
    img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)
    score = predict_lung(model, img)
    scores.append(score)


df = pd.DataFrame({
    "file": [os.path.basename(f) for f in image_files],
    "side": [extract_side(os.path.basename(f)) for f in image_files],
    "score": scores
})

df = df.sort_values(["side","file"])
print(df)


# Save as CSV format
df.to_csv(output_csv, index=False)
print(f"\nSaved: {output_csv}")

## 7. Radiograph-level score

Averages the left and right lung field scores to obtain the score reported in the paper.


In [ ]:
# ---------------------------------------------------------------------------
# Chest radiograph-level (radiographic) BPD likelihood score
# = mean of the left and right lung field-level scores.
# This is the score reported in the paper.
# ---------------------------------------------------------------------------
radiograph_csv = os.path.splitext(output_csv)[0] + "_radiograph_level.csv"

df_r = df.copy()
df_r["radiograph"] = df_r["file"].str.replace(r"_(left|right|unknown)\.png$", "", regex=True)

agg = (
    df_r.groupby("radiograph")["score"]
        .agg(bpd_likelihood_score="mean", n_lung_fields="count")
        .reset_index()
)
agg["above_threshold"] = agg["bpd_likelihood_score"] > SCORE_THRESHOLD

# n_lung_fields < 2 means that one lung field could not be extracted or scored;
# the mean is then based on a single lung field and should be interpreted with care.
print(agg)

agg.to_csv(radiograph_csv, index=False)
print(f"\nSaved: {radiograph_csv}")